# Multiple Regression

Let's grab a small little data set of Blue Book car values:

In [ ]:
from sklearn.preprocessing import StandardScaler
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_excel('http://cdn.sundog-soft.com/Udemy/DataScience/cars.xls')

In [ ]:
# select 'Milege' and 'Price' from df to work with
df_subset = df[['Mileage', 'Price']]
print(df_subset.head())

# Step 1: Define bin edges
bins = np.arange(0, 50000, 10000)

# Step 2: Create a new column with the bin assignment
# df_subset['MileageBin'] = pd.cut(df_subset['Mileage'], bins=bins, right=False)
# Step 3: Group by the new bin column and calculate the mean for each group
grouped_means = df_subset.groupby(pd.cut(df_subset['Mileage'], bins=bins, right=False), observed=False)[['Mileage', 'Price']].mean()

# Step 4: Display result
print(grouped_means)

# plot
grouped_means['Price'].plot()
plt.xlabel('Mileage bins')
plt.ylabel('Mean Price')
plt.show()

# to plot dots.
# grouped_means['Price'].plot(style='o')
# plt.xlabel('Mileage bins')
# plt.ylabel('Mean Price')
# plt.show()

We can use pandas to split up this matrix into the feature vectors we're interested in, and the value we're trying to predict.

Note how we are avoiding the make and model; regressions don't work well with ordinal values, unless you can convert them into some numerical order that makes sense somehow.

Let's scale our feature data into the same range so we can easily compare the coefficients we end up with.

In [ ]:
import statsmodels.api as sm

X = df[['Mileage', 'Cylinder', 'Doors']].copy()
y = df['Price']

scaler = StandardScaler()

# print(X.head())
X[['Mileage', 'Cylinder', 'Doors']] = pd.DataFrame(scaler.fit_transform(X[['Mileage', 'Cylinder', 'Doors']]), columns=X.columns)
# print(X.head())

X = sm.add_constant(X)
model = sm.OLS(y, X)
results = model.fit() #This actually fits the model to your data — meaning it computes the best-fit line by finding the optimal values for the regression coefficients (intercept and slopes).
print(results.summary())

The table of coefficients above gives us the values to plug into an equation of form:
    B0 + B1 * Mileage + B2 * cylinders + B3 * doors
    
In this example, it's pretty clear that the number of cylinders is more important than anything based on the coefficients.

Could we have figured that out earlier?

In [ ]:

y.groupby(df.Doors).mean()

Surprisingly, more doors does not mean a higher price! (Maybe it implies a sport car in some cases?) So it's not surprising that it's pretty useless as a predictor here. This is a very small data set however, so we can't really read much meaning into it.

How would you use this to make an actual prediction? Start by scaling your multiple feature variables into the same scale used to train the model, then just call est.predict() on the scaled features:

In [ ]:

scaled = scaler.transform(pd.DataFrame([[45000, 8, 4]], columns=['Mileage', 'Cylinder', 'Doors']))
print('Scaled:', scaled)

scaled = np.insert(scaled[0], 0, 1)
print('Scaled 2:', scaled)

# predicted = sm.OLS(y, X).fit().predict(scaled)
#
predicted = results.predict(scaled)
print('Predicted price for a car with 45,000 miles, 8 cylinders, and 4 doors: ', predicted)

## Activity

Mess around with the fake input data, and see if you can create a measurable influence of number of doors on price. Have some fun with it - why stop at 4 doors?